# Demo: EEG -> time-frequency image pipeline

This notebook runs the preprocessing + time-frequency image steps of the pipeline on **one healthy-control subject and one MDD subject**, so you can see what the data looks like at each stage without running the full 5-fold training (which takes much longer — use `scripts/run_pipeline.py` for that).

See `README.md` for the plain-English explanation of every step below.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, "..")

import matplotlib.pyplot as plt

from src.config import load_config
from src.preprocessing import discover_recordings, load_and_clean_raw, epoch_raw
from src.tfd_transform import epoch_to_tfd_image

cfg = load_config("../configs/default.yaml")
raw_dir = Path("..") / cfg.data.raw_dir  # notebook runs from notebooks/, config paths are repo-root-relative
recordings = discover_recordings(raw_dir, cfg.data.condition, cfg.data.mdd_prefix, cfg.data.hc_prefix)
hc_rec = next(r for r in recordings if r.subject_id == "H_S1")
mdd_rec = next(r for r in recordings if r.subject_id == "MDD_S1")
print(hc_rec, mdd_rec)

## 1. Load + clean one recording of each class

This applies the bandpass filter, mains notch filter, ICA-based artifact removal, and average re-reference described in the README.

In [ ]:
hc_raw = load_and_clean_raw(hc_rec.path, cfg)
mdd_raw = load_and_clean_raw(mdd_rec.path, cfg)

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
axes[0].plot(hc_raw.times[:1280], hc_raw.get_data(picks="Fz")[0, :1280] * 1e6)
axes[0].set_title("HC (H_S1) - Fz, first 5s, cleaned")
axes[1].plot(mdd_raw.times[:1280], mdd_raw.get_data(picks="Fz")[0, :1280] * 1e6)
axes[1].set_title("MDD (MDD_S1) - Fz, first 5s, cleaned")
axes[1].set_xlabel("seconds")
for ax in axes:
    ax.set_ylabel("uV")
fig.tight_layout()

## 2. Epoch into fixed-length windows

Window length and overlap come from `configs/default.yaml` (`epoching.window_sec`, `epoching.overlap`).

In [ ]:
hc_epochs = epoch_raw(hc_raw, cfg)
mdd_epochs = epoch_raw(mdd_raw, cfg)
print("HC epochs:", hc_epochs.shape, " MDD epochs:", mdd_epochs.shape)

## 3. One epoch -> a time-frequency image

By default this uses the Continuous Wavelet Transform (CWT). The alternative, SPWVD, is also shown for comparison — set `cfg.tfd.transform = "spwvd"` to switch. Both average across all 19 channels into a single image (see README "Assumptions" for why).

In [ ]:
fs = cfg.eeg.sampling_rate_hz
hc_img_cwt = epoch_to_tfd_image(hc_epochs[0], fs, cfg)
mdd_img_cwt = epoch_to_tfd_image(mdd_epochs[0], fs, cfg)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(hc_img_cwt, origin="lower", aspect="auto", cmap="viridis")
axes[0].set_title("HC (H_S1) epoch 0 - CWT")
axes[1].imshow(mdd_img_cwt, origin="lower", aspect="auto", cmap="viridis")
axes[1].set_title("MDD (MDD_S1) epoch 0 - CWT")
fig.tight_layout()

In [ ]:
import copy
cfg_spwvd = copy.deepcopy(cfg)
cfg_spwvd.tfd.transform = "spwvd"
hc_img_spwvd = epoch_to_tfd_image(hc_epochs[0], fs, cfg_spwvd)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(hc_img_cwt, origin="lower", aspect="auto", cmap="viridis")
axes[0].set_title("HC (H_S1) epoch 0 - CWT")
axes[1].imshow(hc_img_spwvd, origin="lower", aspect="auto", cmap="viridis")
axes[1].set_title("HC (H_S1) epoch 0 - SPWVD")
fig.tight_layout()

## Next step

This image (resized to 224x224, replicated to 3 channels) is what feeds the ResNet50/VGG16 classifier. To run full preprocessing + TFD generation + subject-wise 5-fold training on the whole dataset, use:

```
python scripts/run_pipeline.py --config configs/default.yaml
```